In [1]:
import string
import numpy as np
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import confusion_matrix, accuracy_score, precision_score, recall_score, f1_score, classification_report
import pandas as pd

In [2]:
raw_dataset = [
    ("ham", "Go until jurong point, crazy.. Available only in bugis n great world la e buffet."),
    ("ham", "Ok lar... Joking wif u oni..."),
    ("spam", "Free entry in 2 a wkly comp to win FA Cup final tkts 21st May 2005. Text FA to 87121 to receive entry question(std txt rate)T&C's apply 08452810075over18's"),
    ("ham", "U dun say so early hor... U c already then say..."),
    ("ham", "Nah I don't think he goes to usf, he lives around here though"),
    ("spam", "FreeMsg Hey there darling it's been 3 week's now and no word back! I'd like some fun you up for it still? Txt chg to send £1.50 to rcv"),
    ("ham", "Even my brother is not like to speak with me. They treat me like aids patent."),
    ("ham", "As per your request 'Melle Melle (Oru Minnaminunginte Nurungu Vettam)' has been set as your callertune for all Callers. Press *9 to copy your friends Callertune"),
    ("spam", "WINNER!! As a valued network customer you have been selected to receivea £900 prize reward! To claim call 09061701461. Claim code KL341. Valid 12 hours only."),
    ("spam", "Had your mobile 11 months or more? U R entitled to Update to the latest colour mobiles with camera for Free! Call The Mobile Update Co FREE on 08002986030"),
    ("ham", "I'm gonna be home soon and i don't want to talk about this stuff any more tonight, k? I've cried enough today."),
    ("spam", "SIX chances to win CASH! From 100 to 2000 pounds txt> CSH11 and send to 87575. Cost 150p/day, 6days, 16+ TsandCs apply Reply HL 4 info"),
    ("spam", "URGENT! You have won a 1 week FREE membership in our £100,000 Prize Jackpot! Txt the word: CLAIM to No: 81010 T&C [www.dbuk.net] LCCLTD POBOX 4403LDNW1A7RW18"),
    ("ham", "I'VE BEEN SEARCHING FOR YOUR RIGHT WORDS FOR SO LONG IT FUNNY"),
    ("ham", "I HAVE A DATE ON SUNDAY WITH WILL!!"),
    ("spam", "XXXMobileMovieClub: To use your credit, click the WAP link in the next txt message or click here>> http://wap.xxxmobilemovieclub.com?n=QJKGIGHJJGCBL"),
    ("ham", "Oh k...i'm watching here:)"),
    ("ham", "Eh u remember how 2 spell his name... Yes i did. He v naughty make until i v wet."),
    ("ham", "Fine if that's the way u feel. That's the way its gota b"),
    ("spam", "England v Macedonia - dont miss the goals/team news. Txt ur national team to 87077 eg ENGLAND to 87077 Try:WALES, SCOTLAND 4txt/ú1.20 POBOXox36504W45WQ 16+")
]

# PHASE 1: TEXT PREPROCESSING & CLEANING (25% Weight)

In [7]:
def custom_processor(text):
    # Step 1: Convert to lowercase
    text = text.lower()
    
    # Step 2: Remove all standard punctuation marks
    translator = str.maketrans('', '', string.punctuation)
    text = text.translate(translator)
    
    # Step 3: Normalize whitespace
    # Split on whitespace (removes leading/trailing/multiple spaces) and rejoin with single space
    text = ' '.join(text.split())
    
    return text
    
test_examples = [
    "Go until jurong point, crazy.. Available only in bugis n great world la e buffet.",
    "Free entry in 2 a wkly comp to win FA Cup final tkts 21st May 2005.",
    "I'm gonna be home soon and i don't want to talk about this stuff any more tonight, k?"
]
 
for example in test_examples:
    processed = custom_processor(example)
    print(f"Original: {example}")
    print(f"Cleaned:  {processed}")
    print()

Original: Go until jurong point, crazy.. Available only in bugis n great world la e buffet.
Cleaned:  go until jurong point crazy available only in bugis n great world la e buffet

Original: Free entry in 2 a wkly comp to win FA Cup final tkts 21st May 2005.
Cleaned:  free entry in 2 a wkly comp to win fa cup final tkts 21st may 2005

Original: I'm gonna be home soon and i don't want to talk about this stuff any more tonight, k?
Cleaned:  im gonna be home soon and i dont want to talk about this stuff any more tonight k



# PHASE 2: FEATURE ENGINEERING & TOKENIZATION (25% Weight)

In [4]:
print("="*80)
print("PHASE 2: FEATURE ENGINEERING & TOKENIZATION")
print("="*80)
 
# Extract labels and texts from raw dataset
labels = [label for label, _ in raw_dataset]
texts = [text for _, text in raw_dataset]
 
# Initialize CountVectorizer with custom_processor as the preprocessing engine
vectorizer = CountVectorizer(preprocessor=custom_processor, lowercase=False)
 
# Transform cleaned text into numeric Bag-of-Words sparse matrix
X_sparse = vectorizer.fit_transform(texts)
 
print(f"\nCountVectorizer Configuration:")
print(f"  - Custom preprocessor: custom_processor()")
print(f"  - Vocabulary size: {len(vectorizer.get_feature_names_out())} unique words")
print(f"\nSparse Matrix Dimensions:")
print(f"  - Shape: {X_sparse.shape}")
print(f"  - Format: Compressed Sparse Row (CSR)")
print(f"  - Memory efficiency: {X_sparse.nnz} non-zero values out of {X_sparse.shape[0] * X_sparse.shape[1]} total elements")
print(f"  - Sparsity: {100 * (1 - X_sparse.nnz / (X_sparse.shape[0] * X_sparse.shape[1])):.2f}%")
 
print("\n" + "-"*80)
print("STRUCTURAL INTERPRETATION OF SPARSE MATRIX:")
print("-"*80)
print("""
Each Row (index 0 to 19):
  → Represents ONE text document from the dataset
  → Contains the word frequency counts for that specific message
  → Length: Equals vocabulary size (number of unique words across all documents)
 
Each Column (index 0 to vocabulary_size-1):
  → Represents ONE unique WORD from the vocabulary
  → Contains the frequency count of that word across all documents
  → Value at position [i, j] = count of word j in document i
 
Example:
  If Column 5 corresponds to the word "win" and Row 2 value is 3,
  this means document 2 contains the word "win" exactly 3 times.
 
Why Sparse?
  → Most documents don't contain most words
  → Storing only non-zero values saves memory
  → CSR format: Compressed Sparse Row (stores position + value pairs)
""")
 
# Display sample vocabulary and feature names
print("\nSample Vocabulary (first 20 words):")
feature_names = vectorizer.get_feature_names_out()
for idx, word in enumerate(feature_names[:20]):
    print(f"  Column {idx}: '{word}'")
 
# Display a sample document's features
print("\nSample: Document 2 (a spam message) - Word Counts:")
sample_doc_features = X_sparse[2].toarray()[0]
word_counts = [(feature_names[idx], sample_doc_features[idx]) 
               for idx in range(len(feature_names)) if sample_doc_features[idx] > 0]
word_counts.sort(key=lambda x: x[1], reverse=True)
for word, count in word_counts[:10]:
    print(f"  '{word}': {count}")

PHASE 2: FEATURE ENGINEERING & TOKENIZATION

CountVectorizer Configuration:
  - Custom preprocessor: custom_processor()
  - Vocabulary size: 241 unique words

Sparse Matrix Dimensions:
  - Shape: (20, 241)
  - Format: Compressed Sparse Row (CSR)
  - Memory efficiency: 308 non-zero values out of 4820 total elements
  - Sparsity: 93.61%

--------------------------------------------------------------------------------
STRUCTURAL INTERPRETATION OF SPARSE MATRIX:
--------------------------------------------------------------------------------

Each Row (index 0 to 19):
  → Represents ONE text document from the dataset
  → Contains the word frequency counts for that specific message
  → Length: Equals vocabulary size (number of unique words across all documents)

Each Column (index 0 to vocabulary_size-1):
  → Represents ONE unique WORD from the vocabulary
  → Contains the frequency count of that word across all documents
  → Value at position [i, j] = count of word j in document i

Example: